# Notebook 07c — SCTS-v2 Trust Score on Canonical Shared Samples

**Purpose:** Compute per-sample trust scores combining calibration confidence, explanation stability, and conformal coverage. Extends the existing UNSW-only `07_scts_v2.ipynb` to all 3 datasets × 6 model variants using canonical samples from 04c/06v3/05c.

**What is SCTS-v2 (verbatim from existing 07_scts_v2):**

SHAP-Calibrated Trust Score, version 2. A per-alert score in [0, 100] that combines three orthogonal signals:

| Component | What it measures | Source |
|---|---|---|
| **c₁ — Calibration confidence** | Calibrated probability of predicted class | Pre-saved `_test_proba_calibrated.npy` (verified identical to refit hybrid calibrators) |
| **c₂ — Explanation stability** | Worst-case per-sample Jaccard top-10 under adversarial perturbation | `stability_v2_per_sample_jaccard.csv` from 05c |
| **c₃ — Conformal coverage** | Per-sample conformal score at α=0.05 using Mondrian per-class thresholds | Computed inline using Mondrian split-conformal (Vovk 2005, Boström 2020) |

**Combination formula:** SCTS-v2 = (c₁ · c₂ · c₃)^(1/3) · 100

Geometric mean penalises asymmetry: weak in any one component → low SCTS even if others are strong. eps=1e-6 used to avoid log(0).

**Methodology improvements over existing 07_scts_v2:**
1. **Canonical samples:** Uses the 1000 per-dataset shared indices from 04c. Same samples that 06v3 (Krishna) and 05c (stability) used. Eliminates the sample-alignment problem the existing 07 had.
2. **Cleaner conformal split:** All test samples NOT in canonical 1000 serve as the conformal calibration set; canonical 1000 are scored. No sample leakage. More data for the quantile estimate.
3. **All 3 datasets, all 6 model variants:** 18 models total instead of just NSL-or-UNSW.

**Time estimate:** ~5-10 minutes (pure CPU, all data preloaded).

**Outputs:**
- `results/tables/scts_v2_canonical.csv` (per-sample SCTS scores: 18,000 rows)
- `results/tables/scts_v2_per_class_summary.csv` (per-class mean SCTS, c₁, c₂, c₃)
- `results/tables/scts_v2_alpha_sensitivity.csv` (α ∈ {0.05, 0.10, 0.20})
- `results/tables/scts_v2_validation.csv` (SCTS quartile vs accuracy)
- `results/tables/scts_v2_summary.json` (aggregate stats, conformal thresholds)

## 1. Setup

In [18]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [19]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
ALPHAS = [0.05, 0.10, 0.20]
ALPHA_PRIMARY = 0.05  # Used for the main SCTS computation
EPS = 1e-6  # Floor for geometric mean

print(f'Scope: {len(DATASETS)} datasets × {len(MODELS_PER_DATASET)} models = {len(DATASETS) * len(MODELS_PER_DATASET)} model-dataset cells')
print(f'Primary alpha for SCTS: {ALPHA_PRIMARY}')
print(f'Sensitivity alphas: {ALPHAS}')

Scope: 3 datasets × 6 models = 18 model-dataset cells
Primary alpha for SCTS: 0.05
Sensitivity alphas: [0.05, 0.1, 0.2]


## 2. Conformal threshold and per-sample c₃

In [20]:
def split_conformal_threshold(calib_probs, y_calib, alpha):
    """Marginal split-conformal threshold (Romano et al. 2019).
    Nonconformity score s_i = 1 - p_hat(y_i | x_i) using TRUE label.
    Threshold = ceil((n+1)(1-alpha)) / n quantile of scores.
    """
    n = len(y_calib)
    scores = 1.0 - calib_probs[np.arange(n), y_calib]
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)
    return float(np.quantile(scores, q_level))

def mondrian_conformal_thresholds(calib_probs, y_calib, alpha, n_classes=5, min_calib=30):
    """Mondrian per-class conformal thresholds (Vovk 2005, Boström 2020).
    Stratifies by PREDICTED class (deployment-feasible since true class unknown at test time).

    For each predicted class c:
      - Find calibration samples where model predicted class c
      - Compute their nonconformity scores: 1 - p_hat(y_true)
      - Take 95th percentile as threshold for that class
      - Fall back to marginal threshold if n_calib_c < min_calib (defaults to 30)

    Returns: dict {class_idx: threshold}, fallback_classes: list of classes that used marginal
    """
    y_pred = calib_probs.argmax(axis=1)
    marginal_thresh = split_conformal_threshold(calib_probs, y_calib, alpha)

    thresholds = {}
    fallback_classes = []
    n_per_class = {}

    for c in range(n_classes):
        mask_c = (y_pred == c)
        n_c = int(mask_c.sum())
        n_per_class[c] = n_c

        if n_c < min_calib:
            thresholds[c] = marginal_thresh
            fallback_classes.append(c)
        else:
            scores_c = 1.0 - calib_probs[mask_c, :][np.arange(n_c), y_calib[mask_c]]
            q_level = min(np.ceil((n_c + 1) * (1 - alpha)) / n_c, 1.0)
            thresholds[c] = float(np.quantile(scores_c, q_level))

    return thresholds, fallback_classes, n_per_class

def empirical_coverage(test_probs, y_test, thresh):
    """Fraction of test samples where 1 - p_hat(y_true) <= threshold (marginal coverage)."""
    n = len(y_test)
    scores = 1.0 - test_probs[np.arange(n), y_test]
    return float((scores <= thresh).mean())

def empirical_coverage_mondrian(test_probs, y_test, thresholds):
    """Per-sample coverage using Mondrian thresholds (lookup by PREDICTED class)."""
    y_pred = test_probs.argmax(axis=1)
    n = len(y_test)
    scores = 1.0 - test_probs[np.arange(n), y_test]
    sample_thresh = np.array([thresholds[int(p)] for p in y_pred])
    return float((scores <= sample_thresh).mean())

def component_3_mondrian(test_probs, y_pred, thresholds):
    """Per-sample c3 with Mondrian per-class thresholds.
    Threshold for each test sample is determined by its predicted class.
    """
    sample_thresh = np.array([thresholds[int(p)] for p in y_pred], dtype=np.float32)
    s = 1.0 - test_probs[np.arange(len(y_pred)), y_pred]
    c3 = np.clip(1.0 - s / sample_thresh, 0.0, 1.0)
    return c3.astype(np.float32)

def component_3(test_probs, y_pred, thresh):
    """(Legacy) Marginal c3 — single threshold."""
    s = 1.0 - test_probs[np.arange(len(y_pred)), y_pred]
    c3 = np.clip(1.0 - s / thresh, 0.0, 1.0)
    return c3.astype(np.float32)

print('✓ Conformal helpers loaded (marginal + Mondrian per-class)')

✓ Conformal helpers loaded (marginal + Mondrian per-class)


## 3. Load per-sample Jaccard (c₂ source) and compute worst-case per sample

In [21]:
# Load stability per-sample Jaccard (from 05c)
stab_path = Path(REPO) / 'results' / 'tables' / 'stability_v2_per_sample_jaccard.csv'
df_stab = pd.read_csv(stab_path)
print(f'Loaded {stab_path.name}: {len(df_stab)} rows')
print(f'Columns: {df_stab.columns.tolist()}')

# Pivot: for each (dataset, model, sample_position) take MIN Jaccard across 3 perturbations
worst_case = df_stab.groupby(['dataset', 'model', 'sample_position'])['jaccard_top10'].min().reset_index()
worst_case.rename(columns={'jaccard_top10': 'worst_jaccard'}, inplace=True)
print(f'\nWorst-case per sample: {len(worst_case)} rows (expect 18 models × 1000 samples = 18000)')
print(f'Mean worst-case Jaccard: {worst_case["worst_jaccard"].mean():.3f}')
print(f'Min: {worst_case["worst_jaccard"].min():.3f}, Max: {worst_case["worst_jaccard"].max():.3f}')

# Build a lookup: (ds, model) -> array of shape (1000,) of worst-case Jaccards
c2_lookup = {}
for (ds, m), g in worst_case.groupby(['dataset', 'model']):
    g_sorted = g.sort_values('sample_position')
    c2_lookup[(ds, m)] = g_sorted['worst_jaccard'].values.astype(np.float32)

print(f'\nc2_lookup populated for {len(c2_lookup)} (dataset, model) combinations')

Loaded stability_v2_per_sample_jaccard.csv: 54000 rows
Columns: ['dataset', 'model', 'perturbation', 'sample_position', 'true_class', 'jaccard_top10']

Worst-case per sample: 18000 rows (expect 18 models × 1000 samples = 18000)
Mean worst-case Jaccard: 0.478
Min: 0.053, Max: 1.000

c2_lookup populated for 18 (dataset, model) combinations


## 4. Main SCTS computation loop

In [22]:
scts_records = []
alpha_records = []
conformal_meta = {}
validation_records = []
per_class_records = []

t_overall = time.time()
print(f'\n{"="*70}')
print(f'SCTS-v2 computation — {datetime.now().strftime("%H:%M:%S")}')
print(f'{"="*70}\n')

for ds in DATASETS:
    print(f'\n=== {ds} ===')

    # Load canonical indices
    canonical_eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    y_test_full = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')

    # Build conformal calibration mask: all test samples EXCEPT canonical 1000
    n_test = len(y_test_full)
    conformal_calib_mask = np.ones(n_test, dtype=bool)
    conformal_calib_mask[canonical_eval_idx] = False
    n_conformal_calib = conformal_calib_mask.sum()

    y_canonical = y_test_full[canonical_eval_idx]
    y_conformal_calib = y_test_full[conformal_calib_mask]

    print(f'  n_test={n_test}, n_conformal_calib={n_conformal_calib}, n_canonical={len(canonical_eval_idx)}')

    for model_name in MODELS_PER_DATASET:
        # Load pre-saved calibrated probabilities (verified identical to refit hybrid calibrator output)
        prob_path = Path(REPO) / 'calibrators' / ds / f'{model_name}_test_proba_calibrated.npy'
        if not prob_path.exists():
            print(f'  ❌ {model_name}: calibrated probs missing')
            continue
        calibrated_proba_full = np.load(prob_path)

        # Split into conformal calib and canonical (test for SCTS)
        calibrated_proba_conformal = calibrated_proba_full[conformal_calib_mask]
        calibrated_proba_canonical = calibrated_proba_full[canonical_eval_idx]

        # ===== c1: calibration confidence on canonical samples =====
        y_pred_canonical = calibrated_proba_canonical.argmax(axis=1)
        c1 = calibrated_proba_canonical[np.arange(len(y_pred_canonical)), y_pred_canonical].astype(np.float32)

        # ===== c2: worst-case Jaccard (from stability_v2_per_sample_jaccard.csv) =====
        if (ds, model_name) not in c2_lookup:
            print(f'  ⚠ {model_name}: no c2 data, skipping')
            continue
        c2 = c2_lookup[(ds, model_name)]  # already (1000,) float32

        # ===== c3: Mondrian per-class conformal at primary alpha =====
        mondrian_thresh, fallback_classes, n_per_class = mondrian_conformal_thresholds(
            calibrated_proba_conformal, y_conformal_calib, ALPHA_PRIMARY, n_classes=5, min_calib=30
        )
        marginal_thresh = split_conformal_threshold(
            calibrated_proba_conformal, y_conformal_calib, ALPHA_PRIMARY
        )
        c3 = component_3_mondrian(calibrated_proba_canonical, y_pred_canonical, mondrian_thresh)

        # Empirical coverage on canonical 1000 (Mondrian uses per-sample threshold lookup)
        emp_coverage = empirical_coverage_mondrian(calibrated_proba_canonical, y_canonical, mondrian_thresh)

        # ===== SCTS-v2 =====
        geo_mean = (np.clip(c1, EPS, 1.0) * np.clip(c2, EPS, 1.0) * np.clip(c3, EPS, 1.0)) ** (1/3)
        scts = (geo_mean * 100).astype(np.float32)

        # ===== Accuracy on canonical (for validation) =====
        correct = (y_pred_canonical == y_canonical).astype(float)

        # ===== Per-sample records =====
        for i in range(len(scts)):
            scts_records.append({
                'dataset': ds, 'model': model_name,
                'sample_position': i,
                'true_class': int(y_canonical[i]),
                'pred_class': int(y_pred_canonical[i]),
                'correct': int(correct[i]),
                'c1': float(c1[i]),
                'c2': float(c2[i]),
                'c3': float(c3[i]),
                'scts': float(scts[i]),
            })

        # ===== Conformal metadata =====
        conformal_meta[f'{ds}/{model_name}'] = {
            'marginal_threshold_alpha_0.05': float(marginal_thresh),
            'mondrian_thresholds_alpha_0.05': {str(c): float(t) for c, t in mondrian_thresh.items()},
            'fallback_classes': [int(c) for c in fallback_classes],
            'n_calib_per_predicted_class': {str(c): int(n) for c, n in n_per_class.items()},
            'empirical_coverage_on_canonical_mondrian': float(emp_coverage),
            'n_conformal_calibration_samples': int(n_conformal_calib),
        }

        # ===== Alpha sensitivity (Mondrian) =====
        for alpha in ALPHAS:
            mondrian_thresh_a, fallback_a, _ = mondrian_conformal_thresholds(
                calibrated_proba_conformal, y_conformal_calib, alpha, n_classes=5, min_calib=30
            )
            marginal_thresh_a = split_conformal_threshold(
                calibrated_proba_conformal, y_conformal_calib, alpha
            )
            c3_a = component_3_mondrian(calibrated_proba_canonical, y_pred_canonical, mondrian_thresh_a)
            geo_a = (np.clip(c1, EPS, 1.0) * np.clip(c2, EPS, 1.0) * np.clip(c3_a, EPS, 1.0)) ** (1/3)
            scts_a = geo_a * 100
            emp_cov_a = empirical_coverage_mondrian(calibrated_proba_canonical, y_canonical, mondrian_thresh_a)
            alpha_records.append({
                'dataset': ds, 'model': model_name, 'alpha': alpha,
                'mean_threshold_across_classes': float(np.mean(list(mondrian_thresh_a.values()))),
                'marginal_threshold_reference': float(marginal_thresh_a),
                'empirical_coverage_mondrian': float(emp_cov_a),
                'n_fallback_classes': len(fallback_a),
                'mean_scts': float(scts_a.mean()),
                'median_scts': float(np.median(scts_a)),
            })

        # ===== SCTS quartile validation =====
        overall_acc = correct.mean()
        pearson_corr = np.corrcoef(scts, correct)[0, 1] if scts.std() > 1e-9 else 0.0
        for lo, hi in [(0, 25), (25, 50), (50, 75), (75, 101)]:
            mask = (scts >= lo) & (scts < hi)
            n = int(mask.sum())
            acc = float(correct[mask].mean()) if n > 0 else float('nan')
            validation_records.append({
                'dataset': ds, 'model': model_name,
                'scts_bin_low': lo, 'scts_bin_high': hi,
                'n': n, 'accuracy': acc,
                'overall_accuracy': float(overall_acc),
                'pearson_corr_scts_correct': float(pearson_corr),
            })

        # ===== Per-class summary =====
        for c, cname in enumerate(CLASS_NAMES_5):
            mask = y_canonical == c
            if mask.sum() > 0:
                per_class_records.append({
                    'dataset': ds, 'model': model_name,
                    'true_class': cname, 'class_idx': c,
                    'n': int(mask.sum()),
                    'mean_scts': float(scts[mask].mean()),
                    'mean_c1': float(c1[mask].mean()),
                    'mean_c2': float(c2[mask].mean()),
                    'mean_c3': float(c3[mask].mean()),
                    'accuracy': float(correct[mask].mean()),
                })

        print(f'  ▶ {model_name:<22} marg_thr={marginal_thresh:.3f}, '
              f'mondrian=[{",".join(f"{mondrian_thresh[c]:.2f}" for c in range(5))}], '
              f'fb={len(fallback_classes)}/5, emp_cov={emp_coverage:.3f}, '
              f'mean SCTS={scts.mean():.1f}, median={np.median(scts):.1f}, '
              f'acc={overall_acc:.3f}, corr={pearson_corr:+.3f}')

elapsed = (time.time() - t_overall) / 60
print(f'\nTotal time: {elapsed:.1f} min')
print(f'Per-sample records: {len(scts_records)}')
print(f'Per-class records: {len(per_class_records)}')
print(f'Alpha sensitivity records: {len(alpha_records)}')


SCTS-v2 computation — 01:27:47


=== nsl_kdd_v2 ===
  n_test=22544, n_conformal_calib=21544, n_canonical=1000
  ▶ rf_5class_cw           marg_thr=1.000, mondrian=[1.00,0.48,1.00,0.49,1.00], fb=1/5, emp_cov=0.975, mean SCTS=81.8, median=86.3, acc=0.617, corr=-0.032
  ▶ xgb_5class_cw          marg_thr=1.000, mondrian=[1.00,0.29,1.00,0.40,1.00], fb=1/5, emp_cov=0.973, mean SCTS=82.0, median=86.3, acc=0.638, corr=+0.120
  ▶ dnn_5class_cw          marg_thr=1.000, mondrian=[1.00,0.44,0.99,0.38,1.00], fb=1/5, emp_cov=0.962, mean SCTS=75.8, median=75.4, acc=0.627, corr=-0.005
  ▶ rf_5class_smote        marg_thr=1.000, mondrian=[1.00,0.49,1.00,0.46,1.00], fb=1/5, emp_cov=0.977, mean SCTS=74.8, median=75.4, acc=0.615, corr=-0.100
  ▶ xgb_5class_smote       marg_thr=1.000, mondrian=[1.00,0.36,1.00,0.44,1.00], fb=1/5, emp_cov=0.979, mean SCTS=78.7, median=81.4, acc=0.641, corr=+0.146
  ▶ dnn_5class_smote       marg_thr=1.000, mondrian=[1.00,0.46,1.00,1.00,1.00], fb=0/5, emp_cov=0.973, mean SCTS=7

## 5. Build DataFrames and save CSVs

In [23]:
df_scts = pd.DataFrame(scts_records)
df_perclass = pd.DataFrame(per_class_records)
df_alpha = pd.DataFrame(alpha_records)
df_validation = pd.DataFrame(validation_records)

out_dir = Path(REPO) / 'results' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)

df_scts.to_csv(out_dir / 'scts_v2_canonical.csv', index=False)
df_perclass.to_csv(out_dir / 'scts_v2_per_class_summary.csv', index=False)
df_alpha.to_csv(out_dir / 'scts_v2_alpha_sensitivity.csv', index=False)
df_validation.to_csv(out_dir / 'scts_v2_validation.csv', index=False)

print(f'✓ Saved 4 CSVs:')
print(f'  scts_v2_canonical.csv ({len(df_scts)} rows)')
print(f'  scts_v2_per_class_summary.csv ({len(df_perclass)} rows)')
print(f'  scts_v2_alpha_sensitivity.csv ({len(df_alpha)} rows)')
print(f'  scts_v2_validation.csv ({len(df_validation)} rows)')

✓ Saved 4 CSVs:
  scts_v2_canonical.csv (18000 rows)
  scts_v2_per_class_summary.csv (90 rows)
  scts_v2_alpha_sensitivity.csv (54 rows)
  scts_v2_validation.csv (72 rows)


## 6. Print headline findings

In [24]:
print('=' * 70)
print('SCTS-v2 HEADLINE FINDINGS')
print('=' * 70)

# Aggregate by (dataset, model)
df_scts['architecture'] = df_scts['model'].apply(
    lambda m: 'rf' if 'rf' in m else ('xgb' if 'xgb' in m else 'dnn')
)
df_scts['variant'] = df_scts['model'].apply(
    lambda m: 'cw' if 'cw' in m else 'smote'
)

print('\n--- Mean SCTS per (dataset, architecture) averaged across variants ---')
for ds in DATASETS:
    sub = df_scts[df_scts['dataset'] == ds]
    pivot = sub.groupby('architecture')['scts'].mean().round(1)
    print(f'  {ds:<18} {dict(pivot)}')

print('\n--- Validation: SCTS quartile vs accuracy ---')
print('(If SCTS is meaningful, higher SCTS bin should have higher accuracy)')
agg_val = df_validation.groupby(['scts_bin_low', 'scts_bin_high']).agg({
    'n': 'sum',
    'accuracy': 'mean',
}).reset_index()
for _, row in agg_val.iterrows():
    print(f'  SCTS [{int(row["scts_bin_low"]):>3},{int(row["scts_bin_high"]):>3}): '
          f'n={int(row["n"]):>5}, mean_acc={row["accuracy"]:.3f}')

print('\n--- Pearson correlation SCTS vs correctness ---')
corr_summary = df_validation.groupby(['dataset', 'model'])['pearson_corr_scts_correct'].first()
print(f'  Mean correlation across 18 models: {corr_summary.mean():+.3f}')
print(f'  Min: {corr_summary.min():+.3f}, Max: {corr_summary.max():+.3f}')
print(f'  Models with corr > 0: {(corr_summary > 0).sum()}/{len(corr_summary)}')

print('\n--- Alpha sensitivity (mean SCTS across all 18 models) ---')
alpha_pivot = df_alpha.groupby('alpha')[['mean_threshold_across_classes', 'empirical_coverage_mondrian', 'mean_scts']].mean().round(3)
print(alpha_pivot)

print('\n--- Per-class SCTS (mean across 18 models, by true class) ---')
perclass_pivot = df_perclass.groupby('true_class').agg({
    'mean_scts': 'mean', 'mean_c1': 'mean', 'mean_c2': 'mean', 'mean_c3': 'mean',
    'accuracy': 'mean', 'n': 'sum',
}).round(3)
print(perclass_pivot.reindex(CLASS_NAMES_5))

SCTS-v2 HEADLINE FINDINGS

--- Mean SCTS per (dataset, architecture) averaged across variants ---
  nsl_kdd_v2         {'dnn': np.float64(76.5), 'rf': np.float64(78.3), 'xgb': np.float64(80.4)}
  unsw_nb15_v2       {'dnn': np.float64(61.6), 'rf': np.float64(59.4), 'xgb': np.float64(60.9)}
  cic_ids2017_v2     {'dnn': np.float64(69.1), 'rf': np.float64(53.1), 'xgb': np.float64(70.6)}

--- Validation: SCTS quartile vs accuracy ---
(If SCTS is meaningful, higher SCTS bin should have higher accuracy)
  SCTS [  0, 25): n=  752, mean_acc=0.702
  SCTS [ 25, 50): n= 1823, mean_acc=0.615
  SCTS [ 50, 75): n= 6841, mean_acc=0.666
  SCTS [ 75,101): n= 8584, mean_acc=0.860

--- Pearson correlation SCTS vs correctness ---
  Mean correlation across 18 models: +0.280
  Min: -0.100, Max: +0.508
  Models with corr > 0: 15/18

--- Alpha sensitivity (mean SCTS across all 18 models) ---
       mean_threshold_across_classes  empirical_coverage_mondrian  mean_scts
alpha                                      

## 7. Save summary JSON

In [25]:
def to_json_safe(obj):
    """Recursive numpy/tuple-key sanitizer (proven across 06v3 and 05c)."""
    if isinstance(obj, dict):
        new_dict = {}
        for k, v in obj.items():
            if isinstance(k, tuple):
                k = '|'.join(str(x) for x in k)
            elif not isinstance(k, (str, int, float, bool)) and k is not None:
                k = str(k)
            new_dict[k] = to_json_safe(v)
        return new_dict
    elif isinstance(obj, list):
        return [to_json_safe(x) for x in obj]
    elif isinstance(obj, (np.bool_, np.generic)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

summary = {
    'timestamp': datetime.now().isoformat(),
    'method': 'canonical_shared_samples_per_dataset',
    'n_datasets': len(DATASETS),
    'n_models': len(MODELS_PER_DATASET) * len(DATASETS),
    'n_samples_per_model': 1000,
    'primary_alpha': ALPHA_PRIMARY,
    'sensitivity_alphas': ALPHAS,
    'overall_stats': {
        'mean_scts': float(df_scts['scts'].mean()),
        'median_scts': float(df_scts['scts'].median()),
        'min_scts': float(df_scts['scts'].min()),
        'max_scts': float(df_scts['scts'].max()),
        'std_scts': float(df_scts['scts'].std()),
    },
    'mean_components': {
        'c1_calibration': float(df_scts['c1'].mean()),
        'c2_stability': float(df_scts['c2'].mean()),
        'c3_conformal': float(df_scts['c3'].mean()),
    },
    'mean_scts_by_dataset_architecture': df_scts.groupby(['dataset', 'architecture'])['scts'].mean().to_dict(),
    'mean_pearson_corr_scts_correctness': float(
        df_validation.groupby(['dataset', 'model'])['pearson_corr_scts_correct'].first().mean()
    ),
    'conformal_thresholds': conformal_meta,
    'alpha_sensitivity_mean': df_alpha.groupby('alpha')[['mean_threshold_across_classes', 'empirical_coverage_mondrian', 'mean_scts']].mean().to_dict(),
}

summary_safe = to_json_safe(summary)
out_path = Path(REPO) / 'results' / 'tables' / 'scts_v2_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary_safe, f, indent=2)
print(f'✓ Saved: scts_v2_summary.json')

print('\n=== Summary contents ===')
print(json.dumps(summary_safe, indent=2)[:3000])
print('... (truncated)')

✓ Saved: scts_v2_summary.json

=== Summary contents ===
{
  "timestamp": "2026-05-30T01:27:50.438782",
  "method": "canonical_shared_samples_per_dataset",
  "n_datasets": 3,
  "n_models": 18,
  "n_samples_per_model": 1000,
  "primary_alpha": 0.05,
  "sensitivity_alphas": [
    0.05,
    0.1,
    0.2
  ],
  "overall_stats": {
    "mean_scts": 67.76923339111275,
    "median_scts": 74.24734497070312,
    "min_scts": 0.36133965849876404,
    "max_scts": 100.0,
    "std_scts": 19.031792948756877
  },
  "mean_components": {
    "c1_calibration": 0.8935438867459694,
    "c2_stability": 0.4780308663567735,
    "c3_conformal": 0.8226713266840711
  },
  "mean_scts_by_dataset_architecture": {
    "cic_ids2017_v2|dnn": 69.08572135400772,
    "cic_ids2017_v2|rf": 53.11291693438589,
    "cic_ids2017_v2|xgb": 70.57153694072366,
    "nsl_kdd_v2|dnn": 76.5443060348928,
    "nsl_kdd_v2|rf": 78.30975853866339,
    "nsl_kdd_v2|xgb": 80.37415351372957,
    "unsw_nb15_v2|dnn": 61.56935976576805,
    "unsw_n

## 8. Commit

In [26]:
os.chdir(REPO)
!git add notebooks/07c_scts_canonical.ipynb
!git add results/tables/scts_v2_canonical.csv
!git add results/tables/scts_v2_per_class_summary.csv
!git add results/tables/scts_v2_alpha_sensitivity.csv
!git add results/tables/scts_v2_validation.csv
!git add results/tables/scts_v2_summary.json
!git status --short
!git commit -m 'Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)'
!git push origin main

Refresh index: 100% (371/371), done.
 M notebooks/03_nsl_calibration_v2.ipynb
 M notebooks/03d_calibration_bootstrap_cis.ipynb
 M notebooks/03e_refit_hybrid_calibrators.ipynb
 M notebooks/04_shap_analysis.ipynb
 M notebooks/04b_calibration_shap_validation.ipynb
 M notebooks/04c_shap_canonical.ipynb
 M notebooks/05c_stability_canonical.ipynb
 M notebooks/06_krishna_agreement_v3.ipynb
M  notebooks/07c_scts_canonical.ipynb
 M results/tables/krishna_agreement_canonical_summary.json
M  results/tables/scts_v2_alpha_sensitivity.csv
M  results/tables/scts_v2_canonical.csv
M  results/tables/scts_v2_per_class_summary.csv
M  results/tables/scts_v2_summary.json
M  results/tables/scts_v2_validation.csv
?? models/
[main 6258826] Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)
 6 files changed, 12578 insertions(+), 12220 deletions(-)
 rewrit

In [27]:
import json
from pathlib import Path

nb_path = Path('/content/drive/MyDrive/XIDS_Research/xids-research/notebooks/07c_scts_canonical.ipynb')

with open(nb_path) as f:
    nb = json.load(f)

# Look for "mondrian" in the code
found_mondrian = False
for cell in nb['cells']:
    src = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
    if 'mondrian_conformal_thresholds' in src or 'component_3_mondrian' in src:
        found_mondrian = True
        break

print(f'File size: {nb_path.stat().st_size} bytes (expect ~27600)')
print(f'Contains Mondrian functions: {found_mondrian}')
print(f'Total cells: {len(nb["cells"])}')

File size: 39972 bytes (expect ~27600)
Contains Mondrian functions: True
Total cells: 18
